# N17. Held out camera, camera level uncertainty, and the paper's reject check

No training. Loads the checkpoints saved by N14 and N16 and only scores them. About 45 minutes on a GPU.

**1. Development cameras (video_1 to video_5).** Same frozen protocol as N12 to N16.
* Camera level uncertainty: result per camera, leave one camera out, and a bootstrap over cameras next to the usual frame bootstrap.
* Silent failure: classical reader (refused, right, accepted and wrong) against the neural decoders (right, wrong, wrong while over 90 percent sure).
* The paper's reject check: the DeepArUco++ paper says a reading can be discarded when its Hamming distance to the nearest marker is above a threshold, and reports thresholds 3 and 9. The released code has no threshold. Every decoder is scored with no limit, limit 9 and limit 3.

**2. Held out camera (video_6), used once.** The headline results of N12 to N16 are rerun on the held out camera with nothing changed: same arms, same seeds, same severity levels (calibrated on development cameras), same decoding rules. Nothing in this section may be used to change any choice made earlier.

Arms: P (published), B, E (from N14), N, BB, BD (from N16), 5 seeds each.

**Before running: Runtime, Change runtime type, GPU. Then Restart session and Run all.** Only one results workbook is written, to Drive. Nothing is downloaded automatically.


In [1]:
!pip -q install opencv-contrib-python==4.10.0.84 h5py 2>&1 | tail -1

In [2]:
%%writefile deeparuco_port.py
"""
Port the pretrained DeepArUco++ decoder from Keras .h5 to PyTorch, without TensorFlow.

The published decoder (models/dec_new.h5 in AVAuco/deeparuco) is a Keras 2.11 model:

    Input(32, 32, 1)
    Conv2D(8, 5x5, valid) -> PReLU -> MaxPool2D(2)
    Conv2D(4, 3x3, valid) -> PReLU -> MaxPool2D(2)
    Conv2D(1, 1x1, valid, sigmoid)   ->  (6, 6, 1)

Only h5py is needed to read the weights, so the repo's Python 3.9 + TensorFlow
environment is never required for baseline inference.

IMPORTANT PORTING DETAIL
    Keras PReLU defaults to shared_axes=None, so alpha is PER-ELEMENT, not per-channel:
    the stored alphas have shape (28, 28, 8) and (12, 12, 4). torch.nn.PReLU only
    supports per-channel alpha, so using it here silently produces a different model.
    SpatialPReLU below keeps the full alpha tensor.

Layout conversions:
    Keras conv kernel (kh, kw, in, out) -> torch (out, in, kh, kw)
    Keras PReLU alpha (h, w, c)         -> torch (c, h, w)
    Keras 'valid' padding               -> torch padding=0

Usage:
    from deeparuco_port import load_weights, NumpyDecoder, TorchDecoder, verify_port
    verify_port('models/dec_new.h5')          # run this first, once
    dec = TorchDecoder.from_h5('models/dec_new.h5')
    probs = dec(x)                            # x: (N, 1, 32, 32) float in [0, 1]

Upstream code is AGPL-3.0. This module reads a weights file; it does not copy source.
Cite Berral-Soler et al., Image and Vision Computing 152 (2024) 105313.
"""

import numpy as np

DEFAULT_H5 = "models/dec_new.h5"


# --------------------------------------------------------------------------- weights

def load_weights(h5_path=DEFAULT_H5):
    """Read decoder weights from the Keras .h5. Returns a dict of numpy arrays."""
    import h5py

    store = {}
    with h5py.File(h5_path, "r") as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset) and name.startswith("model_weights/"):
                store[name] = np.array(obj)
        f.visititems(visit)

    def pick(layer, suffix):
        hits = [k for k in store if f"/{layer}/" in k and k.endswith(suffix)]
        if len(hits) != 1:
            raise KeyError(f"expected one match for {layer}:{suffix}, got {hits}")
        return store[hits[0]]

    w = {
        "c1_w": pick("conv2d", "kernel:0"),   "c1_b": pick("conv2d", "bias:0"),
        "p1_a": pick("p_re_lu", "alpha:0"),
        "c2_w": pick("conv2d_1", "kernel:0"), "c2_b": pick("conv2d_1", "bias:0"),
        "p2_a": pick("p_re_lu_1", "alpha:0"),
        "c3_w": pick("conv2d_2", "kernel:0"), "c3_b": pick("conv2d_2", "bias:0"),
    }
    expect = {"c1_w": (5,5,1,8), "c1_b": (8,), "p1_a": (28,28,8),
              "c2_w": (3,3,8,4), "c2_b": (4,), "p2_a": (12,12,4),
              "c3_w": (1,1,4,1), "c3_b": (1,)}
    for k, shape in expect.items():
        assert w[k].shape == shape, f"{k}: got {w[k].shape}, expected {shape}"
    return w


# ------------------------------------------------------------------- numpy reference

class NumpyDecoder:
    """Reference implementation. Verified against synthetic markers: 4/5 exact."""

    def __init__(self, weights):
        self.w = weights

    @classmethod
    def from_h5(cls, path=DEFAULT_H5):
        return cls(load_weights(path))

    @staticmethod
    def _conv(x, w, b):
        kh, kw, _, co = w.shape
        H, W, _ = x.shape
        out = np.zeros((H - kh + 1, W - kw + 1, co), np.float32)
        for i in range(out.shape[0]):
            for j in range(out.shape[1]):
                out[i, j] = np.tensordot(
                    x[i:i+kh, j:j+kw, :], w, axes=([0, 1, 2], [0, 1, 2])) + b
        return out

    @staticmethod
    def _pool(x):
        h, w, c = x.shape
        return x.reshape(h // 2, 2, w // 2, 2, c).max(axis=(1, 3))

    def __call__(self, marker32):
        """marker32: (32, 32) float in [0, 1]. Returns (6, 6) bit probabilities."""
        w = self.w
        x = np.asarray(marker32, np.float32)[..., None]
        x = self._conv(x, w["c1_w"], w["c1_b"])
        x = np.where(x > 0, x, w["p1_a"] * x)
        x = self._pool(x)
        x = self._conv(x, w["c2_w"], w["c2_b"])
        x = np.where(x > 0, x, w["p2_a"] * x)
        x = self._pool(x)
        x = self._conv(x, w["c3_w"], w["c3_b"])
        return (1.0 / (1.0 + np.exp(-np.clip(x[..., 0], -60, 60)))).astype(np.float32)


# -------------------------------------------------------------------------- pytorch

def _torch():
    import torch
    import torch.nn as nn
    return torch, nn


class SpatialPReLU:
    """Created via _build so torch is imported lazily. See module docstring."""


def TorchDecoder_factory():
    torch, nn = _torch()

    class _SpatialPReLU(nn.Module):
        """Per-element PReLU, matching Keras PReLU(shared_axes=None)."""
        def __init__(self, alpha_chw):
            super().__init__()
            self.alpha = nn.Parameter(torch.tensor(alpha_chw, dtype=torch.float32))

        def forward(self, x):
            return torch.where(x > 0, x, self.alpha.unsqueeze(0) * x)

    class _TorchDecoder(nn.Module):
        def __init__(self, weights=None):
            super().__init__()
            self.c1 = nn.Conv2d(1, 8, 5)
            self.c2 = nn.Conv2d(8, 4, 3)
            self.c3 = nn.Conv2d(4, 1, 1)
            self.pool = nn.MaxPool2d(2)
            a1 = np.zeros((8, 28, 28), np.float32) if weights is None \
                 else np.transpose(weights["p1_a"], (2, 0, 1))
            a2 = np.zeros((4, 12, 12), np.float32) if weights is None \
                 else np.transpose(weights["p2_a"], (2, 0, 1))
            self.p1, self.p2 = _SpatialPReLU(a1), _SpatialPReLU(a2)
            if weights is not None:
                self.load_keras(weights)

        def load_keras(self, w):
            with torch.no_grad():
                for conv, kw, kb in ((self.c1, "c1_w", "c1_b"),
                                     (self.c2, "c2_w", "c2_b"),
                                     (self.c3, "c3_w", "c3_b")):
                    conv.weight.copy_(torch.tensor(np.transpose(w[kw], (3, 2, 0, 1))))
                    conv.bias.copy_(torch.tensor(w[kb]))

        @classmethod
        def from_h5(cls, path=DEFAULT_H5):
            return cls(load_weights(path))

        def forward(self, x):
            """x: (N, 1, 32, 32) in [0, 1]. Returns (N, 6, 6) bit probabilities."""
            x = self.pool(self.p1(self.c1(x)))
            x = self.pool(self.p2(self.c2(x)))
            return torch.sigmoid(self.c3(x)).squeeze(1)

    return _TorchDecoder


def __getattr__(name):
    if name == "TorchDecoder":
        return TorchDecoder_factory()
    raise AttributeError(name)


# --------------------------------------------------------------------------- verify

def verify_port(h5_path=DEFAULT_H5, n=16, tol=1e-4, verbose=True):
    """Assert the PyTorch port matches the numpy reference. Run once before use.

    The numpy path is validated; the torch path is validated against it here rather
    than assumed. Do not skip this.
    """
    torch, _ = _torch()
    w = load_weights(h5_path)
    ref = NumpyDecoder(w)
    net = TorchDecoder_factory()(w).eval()

    rng = np.random.default_rng(0)
    x = rng.random((n, 32, 32)).astype(np.float32)
    with torch.no_grad():
        got = net(torch.tensor(x).unsqueeze(1)).numpy()
    want = np.stack([ref(xi) for xi in x])

    err = float(np.abs(got - want).max())
    if verbose:
        print(f"port check: {n} random inputs, max abs difference {err:.2e} (tol {tol:g})")
    assert err < tol, f"PyTorch port disagrees with numpy reference (max diff {err:.2e})"
    if verbose:
        print("PyTorch port matches the numpy reference.")
    return err


def decode_ids(probs, id_to_bits, n_markers=250):
    """Hard-decision decode: nearest codeword by Hamming distance over 4 rotations.

    probs: (6, 6) bit probabilities. Returns (marker_id, rotation, hamming).
    Replace with soft log-likelihood decoding to use the confidences.
    """
    obs = (np.asarray(probs) > 0.5).astype(np.uint8)
    best = (None, None, 99)
    for mid in range(n_markers):
        ref = np.asarray(id_to_bits(mid), np.uint8).reshape(6, 6)
        for rot in range(4):
            h = int(np.sum(np.rot90(obs, rot) != ref))
            if h < best[2]:
                best = (mid, rot, h)
    return best


if __name__ == "__main__":
    import sys
    path = sys.argv[1] if len(sys.argv) > 1 else DEFAULT_H5
    w = load_weights(path)
    print("weights loaded:", {k: v.shape for k, v in w.items()})
    try:
        verify_port(path)
    except ImportError:
        print("torch not installed; numpy reference verified only")


Writing deeparuco_port.py


In [3]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'   # deterministic cuBLAS, must be set before torch
import sys, json, glob, zlib, subprocess, time, itertools
import numpy as np, cv2, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE, WORK = '/content/drive/MyDrive/aruco_p2', '/content/work'
except ImportError:
    BASE = os.environ.get('ARUCO_P2_BASE',
                          os.path.join(os.path.expanduser('~'), 'aruco_p2')).replace('\\', '/')
    WORK = f'{BASE}/work'
ART, CACHE = f'{BASE}/artifacts', f'{BASE}/archives'
os.makedirs(WORK, exist_ok=True); sys.path.insert(0, os.getcwd())
assert os.path.exists(f'{ART}/splits.json'), f'{ART}/splits.json missing, run N0 first'

REPO = f'{WORK}/deeparuco'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','https://github.com/AVAuco/deeparuco.git',REPO],check=True)
sys.path.insert(0, REPO)
from impl.aruco import id_to_bits
from deeparuco_port import load_weights, NumpyDecoder

DEV_ = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
splits = json.load(open(f'{ART}/splits.json'))
DEVV, HELDOUT, STRIDE = splits['dev'], splits['heldout'][0], splits['benchmark_stride']
Wk  = load_weights(f'{REPO}/models/dec_new.h5')
CB  = np.stack([np.asarray(id_to_bits(i),np.uint8).reshape(6,6) for i in range(250)])
ROT = np.stack([np.rot90(CB,k,axes=(1,2)).reshape(250,36) for k in range(4)]).astype(np.float32)

SHADOW = f'{WORK}/shadow'; os.makedirs(SHADOW, exist_ok=True)
for i in range(1,7):
    if not os.path.isdir(f'{SHADOW}/video_{i}'):
        tar = f'{CACHE}/video_{i}.tar.gz'
        assert os.path.exists(tar), f'{tar} missing, see the download cell in N0'
        subprocess.run(['tar','-xzf',tar,'-C',SHADOW], check=True)

N0M = json.load(open(f'{ART}/manifest.json'))
for vid in DEVV + [HELDOUT]:
    n = len(glob.glob(f'{SHADOW}/{vid}/*.png'))
    assert n == N0M['shadow_aruco']['frames'][vid], f'{vid}: {n} frames on disk, manifest says other'
print('torch', torch.__version__, '| device', DEV_)
if DEV_.type != 'cuda':
    print('\n*** THIS IS A CPU RUNTIME. One training run takes about 40 minutes here')
    print('*** and about 2 minutes on a GPU. Use Runtime, Change runtime type, T4 or L4,')
    print('*** then rerun every cell from the top before training.')
print('dev:', DEVV, '| held out:', HELDOUT, '| stride', STRIDE)


Mounted at /content/drive
torch 2.14.1+cu130 | device cuda
dev: ['video_1', 'video_2', 'video_3', 'video_4', 'video_5'] | held out: video_6 | stride 6


In [4]:
r_int = lambda v: int(round(v))
VALUES = {'grey':127, 'black':0, 'white':255}

def ordered_corners(xs, ys):
    cx, cy = np.mean(xs), np.mean(ys)
    idx = np.argsort(np.arctan2(xs-cx, ys-cy))
    return [v for i in idx for v in (xs[i], ys[i])]

def warp32(crop, c, t=32):
    dst = np.array([[0,0],[0,t-1],[t-1,t-1],[t-1,0]], np.float32)
    src = np.clip(np.array([[c[0]*63,c[1]*63],[c[2]*63,c[3]*63],
                            [c[4]*63,c[5]*63],[c[6]*63,c[7]*63]]),0,63).astype(np.float32)
    return cv2.warpPerspective(crop, cv2.getPerspectiveTransform(src,dst), (t,t))

def get_corners(m):
    c = m['corners']
    if isinstance(c[0], dict): return np.array([[p['x'],p['y']] for p in c], np.float32)
    return np.asarray(c, np.float32).reshape(4,2)

def cell_occlusion(m32):
    s = 32/8.0; out = np.zeros((6,6), bool)
    for r in range(6):
        for c in range(6):
            out[r,c] = m32[int((r+1)*s):int((r+2)*s), int((c+1)*s):int((c+2)*s)].mean() > 127.5
    return out

def prepare(img, mask, corners, scale=None):
    x, y = corners[:,0], corners[:,1]
    mnx, mxx, mny, mxy = x.min(), x.max(), y.min(), y.max()
    ex, ey = 0.2*(mxx-mnx)+0.5, 0.2*(mxy-mny)+0.5
    mnx, mny = max(mnx-ex,0), max(mny-ey,0)
    mxx = min(mxx+ex, img.shape[1]-1); mxy = min(mxy+ey, img.shape[0]-1)
    if mxx-mnx < 8 or mxy-mny < 8: return None, None
    sl = (slice(r_int(mny), r_int(mxy)), slice(r_int(mnx), r_int(mxx)))
    crop = cv2.resize(img[sl], (64,64)).astype(np.float32)
    lo, hi = (crop.min(), crop.max()) if scale is None else scale
    crop = np.clip((crop - lo)/(hi - lo + 1e-9), 0, 1)
    nc = ordered_corners((x-mnx)/(mxx-mnx), (y-mny)/(mxy-mny))
    m32 = warp32(cv2.resize(mask[sl],(64,64)), nc) if mask.any() else np.zeros((32,32), np.uint8)
    return warp32(crop, nc), cell_occlusion(m32)

def occluder_box(img, corners, level, position, key, mid):
    if level <= 0: return None
    x, y = corners[:,0], corners[:,1]
    x0,x1,y0,y1 = x.min(),x.max(),y.min(),y.max(); w,h = x1-x0, y1-y0
    if position == 'edge':
        side = int(np.random.default_rng(zlib.crc32(f'occlusion|{level}|{key}|{mid}'.encode())).integers(0,4))
        box = [(x0,y0,x0+level*w,y1),(x1-level*w,y0,x1,y1),
               (x0,y0,x1,y0+level*h),(x0,y1-level*h,x1,y1)][side]
    else:
        cx, cy = (x0+x1)/2, (y0+y1)/2
        box = (cx-level*w/2, cy-level*h/2, cx+level*w/2, cy+level*h/2)
    a,b,c,d = [int(round(v)) for v in box]
    a,b = max(a,0),max(b,0); c,d = min(c,img.shape[1]),min(d,img.shape[0])
    return (a,b,c,d) if (c>a and d>b) else None

def apply_occluder(img, box, value):
    if box is None: return img, np.zeros(img.shape[:2], np.uint8)
    if isinstance(value, str): value = VALUES[value]
    a,b,c,d = box
    out = img.copy(); mk = np.zeros(img.shape[:2], np.uint8)
    out[b:d,a:c] = int(value); mk[b:d,a:c] = 255
    return out, mk

def pairs_for(vid):
    out = []
    for f in sorted(glob.glob(f'{SHADOW}/{vid}/*.png')):
        j = f'{SHADOW}/{vid}/corrected_annotations/{os.path.splitext(os.path.basename(f))[0]}.json'
        if os.path.exists(j): out.append((f,j))
    return out

class SpatialPReLU(nn.Module):
    def __init__(s,c,h,w): super().__init__(); s.a = nn.Parameter(torch.zeros(c,h,w))
    def forward(s,x): return torch.where(x>0, x, s.a.unsqueeze(0)*x)

class Decoder(nn.Module):
    """8 and 4 filters, as in the released code. pool is 'max' or 'avg'."""
    def __init__(s, out_ch=1, pool='max'):
        super().__init__()
        s.c1=nn.Conv2d(1,8,5); s.p1=SpatialPReLU(8,28,28)
        s.c2=nn.Conv2d(8,4,3); s.p2=SpatialPReLU(4,12,12)
        s.c3=nn.Conv2d(4,out_ch,1); s.out_ch=out_ch
        s.pool = F.max_pool2d if pool=='max' else F.avg_pool2d
    def forward(s,x):
        x=s.pool(s.p1(s.c1(x)),2); x=s.pool(s.p2(s.c2(x)),2); x=s.c3(x)
        return (x[:,0], x[:,1]) if s.out_ch==2 else (x[:,0], None)
print('pipeline ready')


pipeline ready


In [5]:
def _conv(x,w,b):
    kh,kw,_,_ = w.shape
    return np.einsum('nohckl,klci->nohi', sliding_window_view(x,(kh,kw),axis=(1,2)), w, optimize=True)+b
def _pool(x):
    N,H,Wd,Cc = x.shape; return x.reshape(N,H//2,2,Wd//2,2,Cc).max(axis=(2,4))

def decode_published(x8, chunk=4096):
    out=[]
    for s in range(0,len(x8),chunk):
        x = x8[s:s+chunk].astype(np.float32)[...,None]/255.0
        x = _pool(np.where((y:=_conv(x,Wk['c1_w'],Wk['c1_b']))>0,y,Wk['p1_a']*y))
        x = _pool(np.where((y:=_conv(x,Wk['c2_w'],Wk['c2_b']))>0,y,Wk['p2_a']*y))
        out.append(1/(1+np.exp(-np.clip(_conv(x,Wk['c3_w'],Wk['c3_b'])[...,0],-60,60))))
    return np.concatenate(out).astype(np.float32)

def decode_torch(net, x8, chunk=8192):
    ps, es = [], []
    with torch.no_grad():
        for s in range(0,len(x8),chunk):
            xb = torch.tensor(x8[s:s+chunk], dtype=torch.float32).unsqueeze(1).to(DEV_)/255.0
            lo, el = net(xb)
            ps.append(torch.sigmoid(lo).cpu().numpy())
            es.append(torch.sigmoid(el).cpu().numpy() if el is not None else np.zeros((len(lo),6,6),np.float32))
    return np.concatenate(ps), np.concatenate(es)

def dec_soft(P, Wt=None):
    LP = np.log(np.clip(P,1e-6,1)).reshape(len(P),36)
    LQ = np.log(np.clip(1-P,1e-6,1)).reshape(len(P),36)
    if Wt is not None:
        w = Wt.reshape(len(P),36); LP, LQ = LP*w, LQ*w
    return np.stack([LP@ROT[k].T + LQ@(1-ROT[k]).T for k in range(4)]).max(0).argmax(1)
def hamming_w(P, W=None):
    B = (P.reshape(len(P),36) > 0.5).astype(np.float32)
    W = np.ones_like(B) if W is None else W.reshape(len(P),36).astype(np.float32)
    return np.stack([(B*W)@(1-ROT[k]).T + ((1-B)*W)@ROT[k].T for k in range(4)]).min(0)

def hard_expected(P, ids, W=None):
    """Exact expected accuracy with fair random tie breaking, per marker (0 to 1)."""
    D = hamming_w(P, W); tied = D == D.min(1, keepdims=True)
    return tied[np.arange(len(ids)), ids] / tied.sum(1)
print('decoders ready')


decoders ready


In [6]:
def paired(a_ok, b_ok, groups, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    uf, inv = np.unique(groups, return_inverse=True)
    by = [np.where(inv==i)[0] for i in range(len(uf))]
    obs = a_ok.mean() - b_ok.mean()
    d = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([by[p] for p in rng.integers(0, len(by), len(by))])
        d[i] = a_ok[idx].mean() - b_ok[idx].mean()
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = 2*min((d <= 0).mean(), (d >= 0).mean())
    return float(obs), float(lo), float(hi), max(float(p), 1/n_boot)
from scipy import stats as sst

def welch(a, b):
    """Unpaired difference of means across seeds with a Welch t interval."""
    a, b = np.asarray(a, float), np.asarray(b, float); m = a.mean() - b.mean()
    va, vb = a.var(ddof=1)/len(a), b.var(ddof=1)/len(b); se = np.sqrt(va + vb)
    if se == 0: return float(m), float(m), float(m)
    df = (va + vb)**2 / (va**2/(len(a)-1) + vb**2/(len(b)-1))
    h = sst.t.ppf(0.975, df) * se
    return float(m), float(m - h), float(m + h)
print('statistics ready')


statistics ready


In [7]:
# ------------------------------------------------------------------ border conditions
# The border ring is the outer cell of the 8 by 8 marker; the data region is the inner 6 by 6.
# These are the same definitions and the same five brightnesses as N8 and N11.
INTENS = [0, 64, 127, 192, 255]
BORDER_LEVEL = 0.50            # strip size for data_only and both, as in N8

def region_masks(corners, shape):
    c = corners.mean(axis=0)
    inner = c + (corners - c) * (6.0/8.0)
    m_out = np.zeros(shape, np.uint8); cv2.fillPoly(m_out, [corners.astype(np.int32)], 255)
    m_in  = np.zeros(shape, np.uint8); cv2.fillPoly(m_in,  [inner.astype(np.int32)], 255)
    return cv2.bitwise_and(m_out, cv2.bitwise_not(m_in)), m_in

def make_condition(img, corners, cond, value, key, mid, level=BORDER_LEVEL):
    if cond == 'clean':
        return img, np.zeros(img.shape[:2], np.uint8)
    border, data = region_masks(corners, img.shape[:2])
    if cond == 'border_only':
        paint = border
    else:
        box = occluder_box(img, corners, level, 'edge', key, mid)
        if box is None: return None, None
        a,b,c,d = box
        strip = np.zeros(img.shape[:2], np.uint8); strip[b:d, a:c] = 255
        paint = cv2.bitwise_and(strip, data) if cond == 'data_only' else strip
    out = img.copy(); out[paint > 0] = int(value)
    return out, paint

def crop_scale(img, corners):
    """The (min, max) a clean crop of this marker has. Used by the N11 oracle."""
    x, y = corners[:,0], corners[:,1]
    mnx, mxx, mny, mxy = x.min(), x.max(), y.min(), y.max()
    ex, ey = 0.2*(mxx-mnx)+0.5, 0.2*(mxy-mny)+0.5
    mnx, mny = max(mnx-ex,0), max(mny-ey,0)
    mxx = min(mxx+ex, img.shape[1]-1); mxy = min(mxy+ey, img.shape[0]-1)
    if mxx-mnx < 8 or mxy-mny < 8: return None
    c = cv2.resize(img[r_int(mny):r_int(mxy), r_int(mnx):r_int(mxx)], (64,64)).astype(np.float32)
    return float(c.min()), float(c.max())

def paired_t(a, b):
    d = np.asarray(a, float) - np.asarray(b, float); m = d.mean()
    if len(d) < 2 or d.std(ddof=1) == 0: return float(m), float(m), float(m)
    h = sst.t.ppf(0.975, len(d)-1) * d.std(ddof=1) / np.sqrt(len(d))
    return float(m), float(m-h), float(m+h)

def one_sample_t(v):
    v = np.asarray(v, float); m = v.mean()
    if len(v) < 2 or v.std(ddof=1) == 0: return float(m), float(m), float(m)
    h = sst.t.ppf(0.975, len(v)-1) * v.std(ddof=1) / np.sqrt(len(v))
    return float(m), float(m-h), float(m+h)

def verdict(lo, hi):
    if lo > 0 or hi < 0: return 'differs'
    if lo > -MARGIN and hi < MARGIN: return f'equivalent (within {MARGIN})'
    return 'inconclusive'
print('border conditions and paired statistics ready')


border conditions and paired statistics ready


In [8]:
CKPT     = f'{BASE}/ckpt_n14'; os.makedirs(CKPT, exist_ok=True)   # same folder as N14 and N15
CODE_TAG = 'n14-v1'                     # same training code as N14 and N15
SEEDS    = [0, 1, 2, 3, 4]
MARGIN   = 1.0
EPOCHS, PATIENCE, BS = 200, 20, 256

def signature(arm):
    return json.dumps(dict(ARMS[arm], code=CODE_TAG, epochs=EPOCHS, patience=PATIENCE, bs=BS),
                      sort_keys=True, default=str)

def load_arm(arm, seed):
    path = f'{CKPT}/arm_{arm}_s{seed}.pt'
    if not os.path.exists(path): return None
    ck = torch.load(path, map_location=DEV_)
    assert isinstance(ck, dict) and ck.get('sig') == signature(arm), \
        f'{path} was made with other settings or code. It cannot be compared with this notebook.'
    net = Decoder(ARMS[arm]['out_ch'], ARMS[arm]['pool']).to(DEV_)
    net.load_state_dict(ck['state']); net.eval()
    return net

ARMS = {
 'B':  dict(out_ch=1, mask_bits=False, pool='max', value=127),
 'E':  dict(out_ch=1, mask_bits=False, pool='max', value='random'),
 'BB': dict(out_ch=1, mask_bits=False, pool='max', value='border_random'),
 'BD': dict(out_ch=1, mask_bits=False, pool='max', value='data_random'),
 'N':  dict(out_ch=1, mask_bits=False, pool='max', value='none'),
}   # these five lines must stay exactly as in N14 and N16, or the checkpoints will not load
MODELS = {}
for a in ARMS:
    for s in SEEDS:
        net = load_arm(a, s)
        if net is not None: MODELS[(a, s)] = net
have = sorted({a for a, _ in MODELS})
for a in ARMS:
    assert all((a, s) in MODELS for s in SEEDS), f'arm {a} checkpoints missing from {CKPT}, run N14 and N16 first'
print('loaded arms:', have, f'| {len(MODELS)} checkpoints')
ARM_LIST = ['P'] + have
ARM_SEEDS = {'P': ['pub'], **{a: SEEDS for a in have}}

loaded arms: ['B', 'BB', 'BD', 'E', 'N'] | 25 checkpoints


In [9]:
CELL_PX, CLS_MARGIN = 8, 0.13

def classical_grid(crop_gray, n=6, border=1):
    side = (n + 2*border) * CELL_PX
    img = cv2.resize(crop_gray, (side, side), interpolation=cv2.INTER_NEAREST)
    _, bw = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    g = n + 2*border; m = int(round(CELL_PX*CLS_MARGIN))
    out = np.zeros((g, g), np.uint8)
    for r in range(g):
        for c in range(g):
            cell = bw[r*CELL_PX+m:(r+1)*CELL_PX-m, c*CELL_PX+m:(c+1)*CELL_PX-m]
            out[r, c] = 1 if cv2.countNonZero(cell) > cell.size//2 else 0
    return out

def border_valid(grid, max_white=0.30):
    ring = np.concatenate([grid[0,:], grid[-1,:], grid[1:-1,0], grid[1:-1,-1]])
    return bool(ring.mean() <= max_white)

def classical(x8, ids):
    """Returns accepted (bool) and expected correctness with fair tie breaking (0 to 1)."""
    grids = [classical_grid(x) for x in x8]
    acc   = np.array([border_valid(g) for g in grids])
    obs   = np.stack([g[1:-1,1:-1].reshape(36) for g in grids]).astype(np.float32)
    D = np.stack([obs@(1-ROT[k]).T + (1-obs)@ROT[k].T for k in range(4)]).min(0)
    tied = D == D.min(1, keepdims=True)
    return acc, tied[np.arange(len(ids)), ids] / tied.sum(1)

def render(i, px=200):
    g = np.zeros((8,8), np.uint8); g[1:7,1:7] = CB[i]*255
    return cv2.resize(g, (px,px), interpolation=cv2.INTER_NEAREST)
xs = np.stack([cv2.resize(render(i), (32,32), interpolation=cv2.INTER_AREA) for i in [0,42,100,238,243,249]])
a_, c_ = classical(xs, np.array([0,42,100,238,243,249]))
assert a_.all() and (c_ == 1).all(), 'classical reader failed its self test'
print('classical reader self test: 6 of 6 accepted and correct')

def id_posterior(P):
    """The decoder's own probability for each id, from its 36 outputs, best rotation per id."""
    LP = np.log(np.clip(P,1e-6,1)).reshape(len(P),36); LQ = np.log(np.clip(1-P,1e-6,1)).reshape(len(P),36)
    LL = np.stack([LP@ROT[k].T + LQ@(1-ROT[k]).T for k in range(4)]).max(0)        # (N, 250)
    LL = LL - LL.max(1, keepdims=True); p = np.exp(LL); p /= p.sum(1, keepdims=True)
    return p.argmax(1), p.max(1)

classical reader self test: 6 of 6 accepted and correct


In [10]:
# Frozen protocol: development cameras 1 to 5, every 6th frame. Camera 6 untouched.
COLOURS, TARGETS, POSITION = ['grey', 'white', 'black'], [6, 12, 18], 'edge'
cal = []
for vid in DEVV[:2]:
    for f, j in pairs_for(vid)[::STRIDE*8]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        for m in json.load(open(j))['markers']:
            cal.append((img, get_corners(m), key, int(m['id'])))
        if len(cal) > 400: break
    if len(cal) > 400: break
def mean_cells(level):
    tot=n=0
    for img, cn, key, mid in cal:
        oi, om = apply_occluder(img, occluder_box(img, cn, level, POSITION, key, mid), 'grey')
        _, occ = prepare(oi, om, cn)
        if occ is None: continue
        tot += occ.sum(); n += 1
    return tot/max(n,1)
curve = [(float(L), mean_cells(float(L))) for L in np.arange(0.02, 0.70, 0.02)]
LEVEL_FOR = {T: min(curve, key=lambda lc: abs(lc[1]-T))[0] for T in TARGETS}
print('severity calibration:', {T: round(LEVEL_FOR[T],2) for T in TARGETS}, '(N15: 6 0.3, 12 0.4, 18 0.5)')

KEYS = [('clean',0)] + [('border_only',v) for v in INTENS] + [(c,T) for c in COLOURS for T in TARGETS]

def build_eval(views):
    EV = {k: {'x':[], 'id':[], 'frame':[], 'cam':[], 'occ':[]} for k in KEYS}
    t0 = time.time()
    for vid in views:
        for f, j in pairs_for(vid)[::STRIDE]:
            img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
            key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
            zero = np.zeros(img.shape[:2], np.uint8)
            for m in json.load(open(j))['markers']:
                cn, mid = get_corners(m), int(m['id'])
                xc, _ = prepare(img, zero, cn)
                if xc is None: continue
                items = [(('clean',0), xc, 0)]
                for v in INTENS:
                    oi, om = make_condition(img, cn, 'border_only', v, key, mid)
                    items.append((('border_only',v), prepare(oi, om, cn)[0], 0))
                for T in TARGETS:
                    box = occluder_box(img, cn, LEVEL_FOR[T], POSITION, key, mid)
                    for col in COLOURS:
                        oi, om = apply_occluder(img, box, col)
                        xo, occ = prepare(oi, om, cn)
                        items.append(((col,T), xo, int(occ.sum())))
                for k, x, oc in items:
                    d = EV[k]; d['x'].append((x*255).astype(np.uint8)); d['id'].append(mid)
                    d['frame'].append(key); d['cam'].append(vid); d['occ'].append(oc)
        print(f'  {vid} ({time.time()-t0:.0f}s)')
    for k in EV: EV[k] = {kk: np.array(vv) for kk, vv in EV[k].items()}
    n = len(EV[('clean',0)]['id'])
    assert all(len(EV[k]['id']) == n for k in EV) and n > 1000, f'only {n} markers'
    print(f'\n{n} markers per condition, cameras {sorted(set(EV[("clean",0)]["cam"].tolist()))}')
    print('mean hidden data cells:', {f'{c} {T}': round(float(EV[(c,T)]['occ'].mean()),1) for c in COLOURS[:1] for T in TARGETS})
    return EV

EV = build_eval(DEVV)
assert HELDOUT not in set(EV[('clean',0)]['cam'].tolist()), 'held out camera leaked into the development set'


severity calibration: {6: 0.3, 12: 0.4, 18: 0.5} (N15: 6 0.3, 12 0.4, 18 0.5)
  video_1 (41s)
  video_2 (72s)
  video_3 (111s)
  video_4 (149s)
  video_5 (177s)

12276 markers per condition, cameras ['video_1', 'video_2', 'video_3', 'video_4', 'video_5']
mean hidden data cells: {'grey 6': 5.7, 'grey 12': 11.5, 'grey 18': 18.0}


In [11]:
def score(EV):
    OUT = {}; t0 = time.time()
    for k, d in EV.items():
        acc_, cls_ok = classical(d['x'], d['id'])
        OUT[('classical','-')+k] = dict(accepted=acc_, ok=cls_ok)
        def _entry(P, ids):
            pid, pc = id_posterior(P)
            D = hamming_w(P)                                   # (N, 250), best rotation per id
            return dict(ok=(pid==ids).astype(np.float32), conf=pc,
                        dmin=D.min(1), hard_ok=hard_expected(P, ids))
        OUT[('P','pub')+k] = _entry(decode_published(d['x']), d['id'])
        for (a, s), net in MODELS.items():
            OUT[(a, s)+k] = _entry(decode_torch(net, d['x'])[0], d['id'])
    print(f'scored in {time.time()-t0:.0f}s')
    return OUT

OUT = score(EV)
# the posterior argmax must agree with plain soft decoding, they are the same rule
d = EV[('grey',12)]; assert np.array_equal(id_posterior(decode_published(d['x']))[0], dec_soft(decode_published(d['x'])))
print('check: decoder certainty uses the same id as soft decoding')

scored in 257s
check: decoder certainty uses the same id as soft decoding


## Camera level uncertainty

In [12]:
def cam_values(arm, k):
    """Accuracy per camera; for seeded arms, averaged over seeds first."""
    seeds = ARM_SEEDS[arm]; cams = EV[k]['cam']
    ok = np.mean([OUT[(arm, s)+k]['ok'] for s in seeds], axis=0)
    return {c: 100*float(ok[cams == c].mean()) for c in DEVV}, ok

def cam_boot(ok, cams, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed); uc = np.unique(cams); by = [np.where(cams == c)[0] for c in uc]
    out = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([by[p][rng.integers(0, len(by[p]), len(by[p]))] for p in rng.integers(0, len(by), len(by))])
        out[i] = ok[idx].mean()
    return 100*np.percentile(out, 2.5), 100*np.percentile(out, 97.5)

def frame_boot(ok, frames, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed); uf, inv = np.unique(frames, return_inverse=True)
    by = [np.where(inv == i)[0] for i in range(len(uf))]; out = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([by[p] for p in rng.integers(0, len(by), len(by))]); out[i] = ok[idx].mean()
    return 100*np.percentile(out, 2.5), 100*np.percentile(out, 97.5)

KEY_RESULTS = [('P','border_only',0), ('P','border_only',255), ('N','border_only',255), ('BB','border_only',255), ('P','grey',18), ('P','black',18),
               ('B','grey',18), ('B','black',18), ('E','black',18)] + \
              ([('N','grey',18), ('N','black',18)] if 'N' in ARM_LIST else [])
CAM = []
print('PER CAMERA ACCURACY, leave one camera out, and camera against frame intervals\n')
print(f"{'result':>22}{'all':>8}" + ''.join(f'{c[-1]:>7}' for c in DEVV) +
      f"{'leave one out':>18}{'frame CI':>18}{'camera CI':>18}{'wider':>7}")
for arm, c, T in KEY_RESULTS:
    k = (c, T); per, ok = cam_values(arm, k); cams = EV[k]['cam']
    allv = 100*ok.mean()
    loo = [100*ok[cams != h].mean() for h in DEVV]
    flo, fhi = frame_boot(ok, EV[k]['frame']); clo, chi = cam_boot(ok, cams)
    w = (chi-clo)/max(fhi-flo, 1e-9)
    CAM.append(dict(arm=arm, cond=c, target=T, all=allv, per_camera=per, loo_min=min(loo), loo_max=max(loo),
                    frame_ci=[flo, fhi], camera_ci=[clo, chi], width_ratio=w))
    print(f'{arm+" "+c+" "+str(T):>22}{allv:>8.2f}' + ''.join(f'{per[x]:>7.1f}' for x in DEVV) +
          f'{min(loo):>9.1f} to {max(loo):<5.1f}[{flo:>5.1f},{fhi:>5.1f}]    [{clo:>5.1f},{chi:>5.1f}]{w:>7.1f}')
print(f'\ncamera intervals are {np.median([r["width_ratio"] for r in CAM if r["frame_ci"][1] > r["frame_ci"][0]] or [float("nan")]):.1f} times wider than frame intervals (median)')
print('With only 5 cameras the camera bootstrap is itself rough. Report the leave one out range beside it.')

PER CAMERA ACCURACY, leave one camera out, and camera against frame intervals

                result     all      1      2      3      4      5     leave one out          frame CI         camera CI  wider
       P border_only 0   92.32   91.6   94.5   93.5   88.0   95.0     91.8 to 93.5 [ 91.5, 93.1]    [ 90.0, 94.4]    2.8
     P border_only 255   48.71   49.5   48.4   52.5   44.0   48.9     47.6 to 50.0 [ 46.8, 50.7]    [ 46.0, 51.3]    1.4
     N border_only 255   40.45   44.3   39.8   41.2   35.5   41.1     39.3 to 41.8 [ 38.5, 42.4]    [ 37.6, 43.1]    1.4
    BB border_only 255   95.99   97.3   97.5   97.1   92.1   96.2     95.6 to 97.1 [ 95.4, 96.5]    [ 93.9, 97.4]    2.9
             P grey 18   22.59   24.9   21.1   22.1   21.2   23.3     21.9 to 23.0 [ 21.6, 23.5]    [ 21.2, 24.1]    1.6
            P black 18   44.52   44.6   44.3   39.9   46.8   48.0     43.8 to 45.8 [ 43.4, 45.6]    [ 41.8, 47.0]    2.4
             B grey 18   80.11   81.9   85.0   79.8   73.9   81.0   

## Silent failure

In [13]:
def silent_table(EV, OUT, label):
    print(f'SILENT FAILURE, {label}\n')
    print(f"{'paint':>6} | {'classical: refused':>18}{'right':>8}{'WRONG':>8} |"
          + ''.join(f"{a+': right':>12}{'WRONG':>8}{'WRONG >0.9':>12} |" for a in ARM_LIST))
    SILENT = []
    for k in [('clean',0)] + [('border_only',v) for v in INTENS] + [(c,18) for c in COLOURS]:
        cl = OUT[('classical','-')+k]; acc_, ok_ = cl['accepted'], cl['ok']
        refused = 100*(1-acc_.mean()); right = 100*(acc_*ok_).mean(); wrong = 100*(acc_*(1-ok_)).mean()
        row = dict(cond=k[0], value=k[1], classical_refused=refused, classical_right=right, classical_wrong=wrong)
        line = f'{str(k[1]) if k[0]=="border_only" else k[0]+" "+str(k[1]):>6} | {refused:>17.1f}%{right:>7.1f}%{wrong:>7.1f}% |'
        for a in ARM_LIST:
            oks  = np.array([OUT[(a,s)+k]['ok'].mean() for s in ARM_SEEDS[a]])
            sure = np.array([((1-OUT[(a,s)+k]['ok'])*(OUT[(a,s)+k]['conf']>0.9)).mean() for s in ARM_SEEDS[a]])
            row[f'{a}_right'] = 100*oks.mean(); row[f'{a}_wrong'] = 100*(1-oks.mean()); row[f'{a}_wrong_sure'] = 100*sure.mean()
            line += f'{100*oks.mean():>11.1f}%{100*(1-oks.mean()):>7.1f}%{100*sure.mean():>11.1f}% |'
        SILENT.append(row); print(line)
    print('\nEvery column is a share of ALL markers in that condition.')
    print('classical WRONG = accepted and misread. neural WRONG >0.9 = misread while over 90% sure of the answer.')
    return SILENT

SILENT = silent_table(EV, OUT, 'development cameras 1 to 5')


SILENT FAILURE, development cameras 1 to 5

 paint | classical: refused   right   WRONG |    P: right   WRONG  WRONG >0.9 |    B: right   WRONG  WRONG >0.9 |   BB: right   WRONG  WRONG >0.9 |   BD: right   WRONG  WRONG >0.9 |    E: right   WRONG  WRONG >0.9 |    N: right   WRONG  WRONG >0.9 |
clean 0 |               0.3%   84.7%   15.0% |       99.5%    0.5%        0.4% |       99.6%    0.4%        0.0% |       99.7%    0.3%        0.1% |       99.6%    0.4%        0.1% |       99.8%    0.2%        0.0% |       99.6%    0.4%        0.2% |
     0 |               0.0%   73.1%   26.9% |       92.3%    7.7%        5.0% |       89.2%   10.8%        5.1% |       96.3%    3.7%        1.5% |       95.7%    4.3%        0.1% |       87.9%   12.1%        5.1% |       96.0%    4.0%        1.6% |
    64 |              73.2%   17.1%    9.6% |       78.7%   21.3%       16.7% |       93.4%    6.6%        2.1% |       99.2%    0.8%        0.3% |       92.1%    7.9%        1.7% |       92.6%    7.4%    

## Silent failure with the reject rule described in the paper

In [14]:
def reject_table(EV, OUT, label):
    print(f"SILENT FAILURE WITH THE PAPER'S REJECT RULE, {label}")
    print()
    print('The DeepArUco++ paper rounds the 36 outputs, takes the nearest codeword, and can discard')
    print('a reading whose Hamming distance is above a threshold; it reports thresholds 3 and 9.')
    print('The released code has no threshold. Below: share of ALL markers that are refused,')
    print('accepted and right, and ACCEPTED AND WRONG, with fair tie breaking.\n')
    LIMITS = [None, 9, 3]
    REJ = []
    hdr = ''.join(f"{('no limit' if t is None else 'limit '+str(t)):>30}" for t in LIMITS)
    for a in ARM_LIST:
        print(f'--- {a}' + (' (mean over seeds)' if a != 'P' else ''))
        print(f"{'condition':>14}" + hdr)
        print(f"{'':>14}" + ''.join(f"{'refused':>10}{'right':>10}{'WRONG':>10}" for _ in LIMITS))
        for k in [('clean',0)] + [('border_only',v) for v in INTENS] + [(c,18) for c in COLOURS]:
            line = f'{k[0]+" "+str(k[1]):>14}'
            for t in LIMITS:
                ref_, rig_, wro_ = [], [], []
                for s in ARM_SEEDS[a]:
                    o = OUT[(a, s)+k]
                    acc_ = np.ones_like(o['dmin'], bool) if t is None else (o['dmin'] <= t)
                    ref_.append(1-acc_.mean()); rig_.append((acc_*o['hard_ok']).mean())
                    wro_.append((acc_*(1-o['hard_ok'])).mean())
                r = dict(arm=a, cond=k[0], value=k[1], limit=('none' if t is None else t),
                         refused=100*np.mean(ref_), right=100*np.mean(rig_), wrong=100*np.mean(wro_))
                REJ.append(r)
                line += f"{r['refused']:>9.1f}%{r['right']:>9.1f}%{r['wrong']:>9.1f}%"
            print(line)
        print()
    print('If WRONG stays high at limit 3, the silent failure claim holds even for the pipeline the')
    print('paper describes. If it falls near zero, state the claim for the released code only, and say')
    print('that the reject rule trades the misreads for refusals.')
    return REJ

REJ = reject_table(EV, OUT, 'development cameras 1 to 5')


SILENT FAILURE WITH THE PAPER'S REJECT RULE, development cameras 1 to 5

The DeepArUco++ paper rounds the 36 outputs, takes the nearest codeword, and can discard
a reading whose Hamming distance is above a threshold; it reports thresholds 3 and 9.
The released code has no threshold. Below: share of ALL markers that are refused,
accepted and right, and ACCEPTED AND WRONG, with fair tie breaking.

--- P
     condition                      no limit                       limit 9                       limit 3
                 refused     right     WRONG   refused     right     WRONG   refused     right     WRONG
       clean 0      0.0%     99.5%      0.5%      0.0%     99.5%      0.5%      7.9%     92.1%      0.0%
 border_only 0      0.0%     90.6%      9.4%      2.0%     90.5%      7.4%     36.8%     63.2%      0.0%
border_only 64      0.0%     77.0%     23.0%      2.8%     76.9%     20.3%     58.4%     41.6%      0.0%
border_only 127      0.0%     65.4%     34.6%      3.2%     65.3%     

## Held out camera, used once

Everything below runs on `video_6`, which no earlier choice in the project depended on. The severity levels are the ones calibrated on the development cameras, so the hidden cell counts can differ a little; they are printed. Report these numbers beside the development numbers and do not change anything because of them.


In [15]:
EV_H  = build_eval([HELDOUT])
OUT_H = score(EV_H)
assert set(EV_H[('clean',0)]['cam'].tolist()) == {HELDOUT}


  video_6 (49s)

3456 markers per condition, cameras ['video_6']
mean hidden data cells: {'grey 6': 5.8, 'grey 12': 11.5, 'grey 18': 18.0}
scored in 59s


In [16]:
def acc_seeds(OUTx, arm, k):
    return np.array([100*OUTx[(arm, s)+k]['ok'].mean() for s in ARM_SEEDS[arm]])

def ms(v):
    return f'{v.mean():6.2f}' + (f' +-{v.std(ddof=1):5.2f}' if len(v) > 1 else '        ')

BORDER = [('clean',0)] + [('border_only',v) for v in INTENS]
HEAD = []
for label, EVx, OUTx in [('DEVELOPMENT, cameras 1 to 5', EV, OUT), (f'HELD OUT, {HELDOUT}', EV_H, OUT_H)]:
    print(f'=== {label}: border only, soft accuracy by paint brightness (mean +- sd over seeds)')
    print(f"{'arm':>4}" + ''.join(f"{('clean' if k[0]=='clean' else str(k[1])):>16}" for k in BORDER) + f"{'spread':>16}")
    for a in ARM_LIST:
        vals = {k: acc_seeds(OUTx, a, k) for k in BORDER}
        spread = vals[('border_only',0)] - vals[('border_only',255)]
        print(f'{a:>4}' + ''.join(f'{ms(vals[k]):>16}' for k in BORDER) + f'{ms(spread):>16}')
        for k in BORDER:
            HEAD.append(dict(set=label, arm=a, cond=k[0], value=k[1], mean=float(vals[k].mean()), sd=float(vals[k].std(ddof=1)) if len(vals[k])>1 else 0.0))
        HEAD.append(dict(set=label, arm=a, cond='spread', value='0 minus 255', mean=float(spread.mean()), sd=float(spread.std(ddof=1)) if len(spread)>1 else 0.0))
    print()
    print(f'=== {label}: edge rectangles, soft accuracy')
    print(f"{'arm':>4}" + ''.join(f'{c+" "+str(T):>16}' for c in COLOURS for T in TARGETS))
    for a in ARM_LIST:
        row = ''
        for c in COLOURS:
            for T in TARGETS:
                v = acc_seeds(OUTx, a, (c,T)); row += f'{ms(v):>16}'
                HEAD.append(dict(set=label, arm=a, cond=c, value=T, mean=float(v.mean()), sd=float(v.std(ddof=1)) if len(v)>1 else 0.0))
        print(f'{a:>4}' + row)
    print('mean hidden data cells:', {f'{c} {T}': round(float(EVx[(c,T)]['occ'].mean()),1) for c in COLOURS for T in TARGETS})
    print('\n')


=== DEVELOPMENT, cameras 1 to 5: border only, soft accuracy by paint brightness (mean +- sd over seeds)
 arm           clean               0              64             127             192             255          spread
   P   99.54           92.32           78.68           66.74           56.66           48.71           43.61        
   B   99.60 +- 0.59   89.15 +- 2.03   93.45 +- 3.37   86.89 +- 6.74   81.62 +- 7.59   77.12 +- 7.63   12.03 +- 9.36
  BB   99.68 +- 0.12   96.34 +- 1.38   99.17 +- 0.50   99.20 +- 0.59   98.26 +- 1.37   95.99 +- 2.84    0.35 +- 3.61
  BD   99.64 +- 0.05   95.69 +- 2.91   92.13 +- 2.41   83.31 +- 3.40   77.10 +- 3.34   72.79 +- 3.94   22.89 +- 3.97
   E   99.82 +- 0.15   87.85 +- 3.02   92.61 +- 2.94   83.79 +- 4.54   77.87 +- 4.16   73.78 +- 4.27   14.07 +- 5.08
   N   99.56 +- 0.25   96.02 +- 0.87   71.81 +- 2.29   59.36 +- 3.22   48.91 +- 4.27   40.45 +- 5.18   55.58 +- 5.83

=== DEVELOPMENT, cameras 1 to 5: edge rectangles, soft accuracy
 arm        

In [17]:
def spread_seeds(OUTx, a):
    return acc_seeds(OUTx, a, ('border_only',0)) - acc_seeds(OUTx, a, ('border_only',255))

PAIRS = []
def show(label, name, d_dev, d_held):
    for set_, d in (('dev', d_dev), ('held out', d_held)):
        m, lo, hi = d
        print(f'  {name:<34}{set_:>9}{m:>9.2f}   [{lo:>7.2f}, {hi:>7.2f}]   {verdict(lo, hi)}')
        PAIRS.append(dict(comparison=name, set=set_, diff=m, lo=lo, hi=hi, verdict=verdict(lo, hi)))

print('PAIRED DIFFERENCES OVER SEEDS, development against held out\n')
print('Q1. border training (spread, BB minus N; negative means BB removes the effect)')
show('', 'BB minus N, spread', paired_t(spread_seeds(OUT,'BB'), spread_seeds(OUT,'N')),
                             paired_t(spread_seeds(OUT_H,'BB'), spread_seeds(OUT_H,'N')))
show('', 'BB minus N, clean', paired_t(acc_seeds(OUT,'BB',('clean',0)), acc_seeds(OUT,'N',('clean',0))),
                            paired_t(acc_seeds(OUT_H,'BB',('clean',0)), acc_seeds(OUT_H,'N',('clean',0))))
print('\npublished decoder spread minus N spread (interval over N seeds)')
for OUTx, set_ in ((OUT, 'dev'), (OUT_H, 'held out')):
    p = spread_seeds(OUTx, 'P')[0]; m, lo, hi = one_sample_t(p - spread_seeds(OUTx, 'N'))
    print(f'  {"P minus N, spread":<34}{set_:>9}{m:>9.2f}   [{lo:>7.2f}, {hi:>7.2f}]   {verdict(lo, hi)}')
    PAIRS.append(dict(comparison='P minus N, spread', set=set_, diff=m, lo=lo, hi=hi, verdict=verdict(lo, hi)))
print('\nQ2. N minus B (positive means the decoder without rectangles is better)')
for c in COLOURS:
    for T in TARGETS:
        show('', f'N minus B, {c} {T}', paired_t(acc_seeds(OUT,'N',(c,T)), acc_seeds(OUT,'B',(c,T))),
                                       paired_t(acc_seeds(OUT_H,'N',(c,T)), acc_seeds(OUT_H,'B',(c,T))))
print('\nBB minus N on black rectangles (the one cost of border training)')
for T in TARGETS:
    show('', f'BB minus N, black {T}', paired_t(acc_seeds(OUT,'BB',('black',T)), acc_seeds(OUT,'N',('black',T))),
                                      paired_t(acc_seeds(OUT_H,'BB',('black',T)), acc_seeds(OUT_H,'N',('black',T))))
print('\nA finding is confirmed on the held out camera when the held out verdict has the same sign.')
print('With one camera, a held out interval that is inconclusive is not a contradiction.')


PAIRED DIFFERENCES OVER SEEDS, development against held out

Q1. border training (spread, BB minus N; negative means BB removes the effect)
  BB minus N, spread                      dev   -55.22   [ -63.93,  -46.51]   differs
  BB minus N, spread                 held out   -56.90   [ -65.11,  -48.69]   differs
  BB minus N, clean                       dev     0.12   [  -0.19,    0.42]   equivalent (within 1.0)
  BB minus N, clean                  held out     0.25   [  -0.15,    0.65]   equivalent (within 1.0)

published decoder spread minus N spread (interval over N seeds)
  P minus N, spread                       dev   -11.97   [ -19.21,   -4.73]   differs
  P minus N, spread                  held out   -17.52   [ -25.18,   -9.86]   differs

Q2. N minus B (positive means the decoder without rectangles is better)
  N minus B, grey 6                       dev   -24.64   [ -30.08,  -19.21]   differs
  N minus B, grey 6                  held out   -28.08   [ -33.79,  -22.38]   differs
  

In [18]:
SILENT_H = silent_table(EV_H, OUT_H, f'held out camera {HELDOUT}')
print()
REJ_H = reject_table(EV_H, OUT_H, f'held out camera {HELDOUT}')


SILENT FAILURE, held out camera video_6

 paint | classical: refused   right   WRONG |    P: right   WRONG  WRONG >0.9 |    B: right   WRONG  WRONG >0.9 |   BB: right   WRONG  WRONG >0.9 |   BD: right   WRONG  WRONG >0.9 |    E: right   WRONG  WRONG >0.9 |    N: right   WRONG  WRONG >0.9 |
clean 0 |               2.0%   86.9%   11.1% |       98.4%    1.6%        0.8% |       99.5%    0.5%        0.1% |       99.5%    0.5%        0.2% |       99.4%    0.6%        0.2% |       99.7%    0.3%        0.0% |       99.3%    0.7%        0.4% |
     0 |               0.0%   67.5%   32.5% |       84.5%   15.5%       10.4% |       80.4%   19.6%        9.0% |       94.3%    5.7%        2.3% |       93.9%    6.1%        0.2% |       78.7%   21.3%        9.3% |       94.2%    5.8%        2.4% |
    64 |              71.7%   19.2%    9.1% |       73.8%   26.2%       19.7% |       93.8%    6.2%        2.7% |       99.2%    0.8%        0.3% |       91.3%    8.7%        1.6% |       93.2%    6.8%       

In [19]:
res = dict(protocol=dict(dev_views=DEVV, heldout=HELDOUT, stride=STRIDE,
                         markers_dev=int(len(EV[('clean',0)]['id'])), markers_heldout=int(len(EV_H[('clean',0)]['id'])),
                         heldout_used=True, note='held out camera used once, after all choices were fixed'),
           arms=ARM_LIST, cameras=CAM, headline=HEAD, paired=PAIRS,
           silent_failure_dev=SILENT, reject_rule_dev=REJ, silent_failure_heldout=SILENT_H, reject_rule_heldout=REJ_H,
           levels={str(k): v for k, v in LEVEL_FOR.items()})
with open(f'{ART}/n17_dev_and_heldout.json', 'w') as fh: json.dump(res, fh, indent=1, default=float)
print('saved', f'{ART}/n17_dev_and_heldout.json')

saved /content/drive/MyDrive/aruco_p2/artifacts/n17_dev_and_heldout.json


## Results workbook

In [20]:
# ---------------------------------------------------------------------------------
# Collect every saved result (all .json files in the artifacts folder) into one Excel
# workbook. One tab per file, plus an INDEX tab listing every file and a COVERAGE tab
# showing which notebooks have no saved results yet. Safe to rerun at any time.
# ---------------------------------------------------------------------------------
import re, glob, json, math, datetime
import numpy as np, pandas as pd
from openpyxl.styles import Font

XLSX = f'{ART}/all_results.xlsx'
NOTEBOOKS = list(range(0, 18))          # notebooks expected to have saved something
MAX_ROWS = 1_000_000

# how to name the parts of keys written as 'a|b|c'
KNOWN = {
    'accuracy':        ['arm', 'seed', 'colour', 'target'],
    'accuracy_soft':   ['arm', 'seed', 'colour', 'target'],
    'seconds':         ['arm', 'seed'],
    'cap':             ['seed', 'eps', 'colour', 'target', 'split'],
    'n13_unified_dev': ['arm', 'seed', 'colour', 'target', 'decoder'],
    'n14_unified_dev': ['arm', 'seed', 'colour', 'target', 'decoder'],
    'n15_unified_dev': ['arm', 'seed', 'colour', 'target', 'decoder'],
    'n16_unified_dev': ['arm', 'seed', 'colour', 'target', 'decoder'],
}
NOTES = {   # results that later notebooks replaced
    'n13_seeds_dev':   'superseded by N14 (N13 mixed checkpoints from two code versions)',
    'n13_unified_dev': 'superseded by N14 (N13 mixed checkpoints from two code versions)',
}

def is_scalar(x): return x is None or isinstance(x, (str, int, float, bool))

def num(s):
    try: return pd.to_numeric(s)
    except (ValueError, TypeError): return s

def cols_for(name, stem, n):
    for part in [p for p in reversed(name.split('.'))] + [stem]:
        if part in KNOWN and len(KNOWN[part]) == n: return KNOWN[part]
    return [f'key{i+1}' for i in range(n)]

def split_keys(d, name, stem):
    rows = [str(k).split('|') for k in d]
    n = max(len(r) for r in rows)
    names = cols_for(name, stem, n)
    df = pd.DataFrame([r + [None]*(n-len(r)) for r in rows], columns=names)
    for c in names: df[c] = num(df[c])
    df['value'] = list(d.values())
    return df

def tables(obj, name, stem):
    """Yield (title, DataFrame) pairs for any JSON value."""
    if isinstance(obj, list):
        if obj and all(isinstance(x, dict) for x in obj):
            yield name, pd.json_normalize(obj, sep='.')
        elif all(is_scalar(x) for x in obj):
            yield name, pd.DataFrame({name.split('.')[-1] or 'value': obj})
        else:
            yield name, pd.DataFrame(obj)
        return
    if not isinstance(obj, dict):
        yield name, pd.DataFrame({'value': [obj]}); return

    scal = {k: v for k, v in obj.items() if is_scalar(v)}
    rest = {k: v for k, v in obj.items() if not is_scalar(v)}
    if scal:
        if len(scal) >= 3 and all('|' in str(k) for k in scal):
            yield name, split_keys(scal, name, stem)
        else:
            yield name, pd.DataFrame({'key': [str(k) for k in scal], 'value': list(scal.values())})
    # a dict of small flat dicts (e.g. arm settings) becomes one table, one row each
    flat_dicts = {k: v for k, v in rest.items() if isinstance(v, dict) and v
                  and all(is_scalar(x) for x in v.values()) and not any('|' in str(x) for x in v)}
    if len(flat_dicts) >= 2:
        df = pd.DataFrame.from_dict(flat_dicts, orient='index')
        col = 'name' if 'name' not in df.columns else 'item'   # manifest.json has a 'name' field
        df.insert(0, col, df.index.astype(str)); yield name, df.reset_index(drop=True)
        rest = {k: v for k, v in rest.items() if k not in flat_dicts}
    for k, v in rest.items():
        yield from tables(v, f'{name}.{k}' if name else str(k), stem)

def clean(df):
    df = df.copy()
    for c in df.columns:
        if df[c].map(lambda x: isinstance(x, (list, dict))).any():
            df[c] = df[c].map(lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)
        if df[c].dtype.kind == 'f':
            df[c] = df[c].where(np.isfinite(df[c]), None)
    df.columns = [str(c) for c in df.columns]
    return df.head(MAX_ROWS)

def sheet_name(stem, used):
    s = re.sub(r'[\[\]:*?/\\]', '_', stem)[:31] or 'sheet'
    base, i = s, 2
    while s.lower() in used:
        s = f'{base[:28]}_{i}'; i += 1
    used.add(s.lower()); return s

files = sorted(set(glob.glob(f'{ART}/**/*.json', recursive=True)) | set(glob.glob(f'{BASE}/*.json')))
def nb_of(path):
    if os.path.basename(path).lower() in ('manifest.json', 'splits.json'): return 0
    m = re.match(r'^n(\d+)', os.path.basename(path).lower())
    return int(m.group(1)) if m else None
files.sort(key=lambda p: (nb_of(p) if nb_of(p) is not None else -1, os.path.basename(p)))
print(f'found {len(files)} result files under {ART}\n')

index, used, bad = [], {'index', 'coverage'}, []
with pd.ExcelWriter(XLSX, engine='openpyxl') as xw:
    pd.DataFrame({'building': ['...']}).to_excel(xw, sheet_name='INDEX', index=False)
    pd.DataFrame({'building': ['...']}).to_excel(xw, sheet_name='COVERAGE', index=False)
    for path in files:
        stem = os.path.splitext(os.path.basename(path))[0]
        try:
            obj = json.load(open(path))
            parts = [(t, clean(df)) for t, df in tables(obj, '', stem) if len(df.columns)]
        except Exception as e:
            bad.append((path, repr(e))); print(f'  could not read {path}: {e!r}'); continue
        sh = sheet_name(stem, used); row = 0
        for title, df in parts:
            df.to_excel(xw, sheet_name=sh, startrow=row + 1, index=False)
            ws = xw.sheets[sh]
            ws.cell(row=row + 1, column=1, value=title or stem).font = Font(bold=True)
            row += len(df) + 4
        index.append(dict(notebook=f'N{nb_of(path)}' if nb_of(path) is not None else 'setup',
                          sheet=sh, file=os.path.relpath(path, BASE),
                          modified=datetime.datetime.fromtimestamp(os.path.getmtime(path)).strftime('%Y-%m-%d %H:%M'),
                          size_kb=round(os.path.getsize(path)/1024, 1), tables=len(parts),
                          rows=int(sum(len(d) for _, d in parts)), note=NOTES.get(stem, '')))
        print(f'  {sh:<32} {len(parts):>3} tables {index[-1]["rows"]:>8} rows   {index[-1]["file"]}')

    idx = pd.DataFrame(index)
    have = {nb_of(p) for p in files if nb_of(p) is not None}
    cov = pd.DataFrame([dict(notebook=f'N{n}', files=int(sum(1 for p in files if nb_of(p) == n)),
                             status='saved' if n in have else
                                    'NOTHING SAVED: rerun this notebook, or its results exist only as printed output')
                        for n in NOTEBOOKS])
    idx.to_excel(xw, sheet_name='INDEX', index=False)
    cov.to_excel(xw, sheet_name='COVERAGE', index=False)
    for name in ('INDEX', 'COVERAGE'):
        ws = xw.sheets[name]
        for cell in ws[1]: cell.font = Font(bold=True)
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(80, max(10, max(len(str(c.value or '')) for c in col) + 2))

print(f'\nsaved {XLSX}')
print('\nCOVERAGE')
print(cov.to_string(index=False))
if bad: print('\nfiles that could not be read:', *bad, sep='\n  ')
print('Download it from Drive: MyDrive/aruco_p2/artifacts/all_results.xlsx')


found 26 result files under /content/drive/MyDrive/aruco_p2/artifacts

  manifest                          14 tables       56 rows   artifacts/manifest.json
  splits                             3 tables        8 rows   artifacts/splits.json
  n1a_headroom                       3 tables       23 rows   artifacts/n1a_headroom.json
  n2_occlusion                       6 tables       32 rows   artifacts/n2_occlusion.json
  n3_results                         4 tables       21 rows   artifacts/n3_results.json
  n3b_results                        3 tables       23 rows   artifacts/n3b_results.json
  n4_controlled                    188 tables      506 rows   artifacts/n4_controlled.json
  n5_significance                    4 tables      109 rows   artifacts/n5_significance.json
  n6_mechanism_mitigation            4 tables       20 rows   artifacts/n6_mechanism_mitigation.json
  n7_error_localisation              4 tables       23 rows   artifacts/n7_error_localisation.json
  n8_border_causal